# SHAP and LIME comparison

Experiments for the paper. Run the cells in order.

Outputs are written to `outputs/` and are limited to the tables and figures the paper needs.

## 1. Imports

In [ ]:
from pathlib import Path
from time import perf_counter
from itertools import combinations

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, mannwhitneyu
from xgboost import XGBClassifier

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 2. Configuration

Every value the paper reports is set here, so the methodology section can be written from this cell.

In [ ]:
OUTPUT_DIR = Path("outputs")
TELCO_KAGGLE_DATASET = "blastchar/telco-customer-churn"

RANDOM_SEED = 42
TEST_SET_FRACTION = 0.2
ADULT_SAMPLE_ROWS = 12000

# SHAP and LIME explain this same set of test instances
EXPLANATION_INSTANCE_COUNT = 100

# LinearSHAP reference distribution. TreeSHAP uses the tree path dependent
# perturbation and therefore needs no background sample.
SHAP_BACKGROUND_SIZE = 100

# LIME settings, stated explicitly rather than left at library defaults
LIME_PERTURBATION_SAMPLES = 2000
LIME_KERNEL_WIDTH = None  # None gives the library default of sqrt(n_features) * 0.75

# Stability protocol
STABILITY_INSTANCE_COUNT = 30
STABILITY_RUN_COUNT = 20
STABILITY_TOP_K = 10

BOOTSTRAP_REPLICATES = 1000
TOP_FEATURES_REPORTED = 5

MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost"]

OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Data loading

In [ ]:
def standardise_column_names(frame):
    frame = frame.copy()
    frame.columns = (
        frame.columns.str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return frame


def load_telco_dataset():
    download_path = Path(kagglehub.dataset_download(TELCO_KAGGLE_DATASET))
    csv_path = sorted(download_path.glob("*.csv"))[0]
    raw = pd.read_csv(csv_path)

    raw.columns = raw.columns.str.strip()
    raw = raw.drop_duplicates()
    for column in raw.select_dtypes(include="object").columns:
        raw[column] = raw[column].str.strip()

    raw["TotalCharges"] = pd.to_numeric(raw["TotalCharges"], errors="coerce")
    raw["TotalCharges"] = raw["TotalCharges"].fillna(raw["TotalCharges"].median())

    encoded = raw.drop(columns=["customerID"]).copy()
    for column in ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]:
        encoded[column] = encoded[column].map({"Yes": 1, "No": 0})
    encoded["gender"] = encoded["gender"].map({"Male": 1, "Female": 0})

    encoded = pd.get_dummies(
        encoded,
        columns=[
            "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
            "Contract", "PaymentMethod",
        ],
        dtype=int,
    )
    return standardise_column_names(encoded), "Churn"


def load_adult_dataset():
    raw = fetch_openml("adult", version=2, as_frame=True).frame.copy()
    raw = standardise_column_names(raw)
    raw = raw.replace("?", pd.NA).dropna()

    # fnlwgt is a census sampling weight, not a predictor
    if "fnlwgt" in raw.columns:
        raw = raw.drop(columns=["fnlwgt"])

    target = "class"
    raw[target] = raw[target].astype(str).str.strip().str.replace(".", "", regex=False)
    raw[target] = raw[target].map({"<=50K": 0, ">50K": 1})
    raw = raw.dropna(subset=[target])
    raw[target] = raw[target].astype(int)

    for column in raw.select_dtypes(include=["category", "object"]).columns:
        raw[column] = raw[column].astype(str).str.strip()

    categorical_columns = [
        column for column in raw.select_dtypes(include=["category", "object"]).columns
        if column != target
    ]
    encoded = pd.get_dummies(raw, columns=categorical_columns, dtype=int)
    encoded = standardise_column_names(encoded)

    if len(encoded) > ADULT_SAMPLE_ROWS:
        encoded, _ = train_test_split(
            encoded,
            train_size=ADULT_SAMPLE_ROWS,
            random_state=RANDOM_SEED,
            stratify=encoded[target],
        )
        encoded = encoded.reset_index(drop=True)
    return encoded, target

## 4. Preprocessing and models

In [ ]:
def split_and_scale(encoded_frame, target_column):
    """Split first, then standardise continuous columns using training statistics only."""
    features = encoded_frame.drop(columns=[target_column])
    target = encoded_frame[target_column]

    features_train, features_test, target_train, target_test = train_test_split(
        features, target,
        test_size=TEST_SET_FRACTION,
        random_state=RANDOM_SEED,
        stratify=target,
    )

    continuous_columns = [c for c in features_train.columns if features_train[c].nunique() > 2]
    if continuous_columns:
        scaler = StandardScaler()
        features_train = features_train.copy()
        features_test = features_test.copy()
        features_train[continuous_columns] = scaler.fit_transform(features_train[continuous_columns])
        features_test[continuous_columns] = scaler.transform(features_test[continuous_columns])

    return features_train, features_test, target_train, target_test, continuous_columns


def build_models(target_train):
    """All three models handle class imbalance the same way."""
    negative_count = int((target_train == 0).sum())
    positive_count = int((target_train == 1).sum())
    positive_weight = negative_count / positive_count if positive_count else 1.0

    return {
        "Logistic Regression": LogisticRegression(
            max_iter=5000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_SEED,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=150,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=150,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            scale_pos_weight=positive_weight,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        ),
    }


def score_predictions(target_test, predictions):
    return {
        "accuracy": accuracy_score(target_test, predictions),
        "precision": precision_score(target_test, predictions, zero_division=0),
        "recall": recall_score(target_test, predictions, zero_division=0),
        "f1_score": f1_score(target_test, predictions, zero_division=0),
    }

## 5. SHAP

In [ ]:
def extract_positive_class_values(shap_output):
    values = shap_output.values if hasattr(shap_output, "values") else shap_output
    if isinstance(values, list):
        return values[1]
    if values.ndim == 3:
        return values[:, :, 1]
    return values


def compute_shap_feature_ranking(model_name, model, features_train, explanation_instances):
    """Global ranking from mean absolute SHAP value over the shared instance set."""
    start_time = perf_counter()

    if model_name == "Logistic Regression":
        background = features_train.sample(
            min(SHAP_BACKGROUND_SIZE, len(features_train)), random_state=RANDOM_SEED
        )
        explainer = shap.LinearExplainer(model, background)
    else:
        explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")

    shap_values = extract_positive_class_values(explainer(explanation_instances))

    ranking = (
        pd.DataFrame({
            "feature": explanation_instances.columns,
            "shap_importance": np.abs(shap_values).mean(axis=0),
        })
        .sort_values("shap_importance", ascending=False)
        .reset_index(drop=True)
    )
    ranking["shap_rank"] = ranking.index + 1
    return ranking, perf_counter() - start_time

## 6. LIME

In [ ]:
def build_lime_explainer(features_train, seed):
    return LimeTabularExplainer(
        training_data=features_train.to_numpy(),
        feature_names=features_train.columns.tolist(),
        class_names=["No", "Yes"],
        mode="classification",
        discretize_continuous=True,
        kernel_width=LIME_KERNEL_WIDTH,
        random_state=seed,
    )


def make_prediction_function(model, feature_names):
    def predict_probabilities(values):
        return model.predict_proba(pd.DataFrame(values, columns=feature_names))
    return predict_probabilities


def compute_lime_feature_ranking(model, features_train, explanation_instances):
    """LIME scores every feature, on the same instances SHAP explained."""
    start_time = perf_counter()
    feature_names = features_train.columns.tolist()
    predict_probabilities = make_prediction_function(model, feature_names)
    explainer = build_lime_explainer(features_train, RANDOM_SEED)

    records = []
    for instance_index, (_, instance) in enumerate(explanation_instances.iterrows()):
        explanation = explainer.explain_instance(
            instance.to_numpy(),
            predict_probabilities,
            num_features=len(feature_names),
            num_samples=LIME_PERTURBATION_SAMPLES,
        )
        # as_map returns feature indices, so no name matching is required
        for feature_index, weight in explanation.as_map()[1]:
            records.append({
                "instance": instance_index,
                "feature": feature_names[feature_index],
                "lime_abs_weight": abs(weight),
            })

    ranking = (
        pd.DataFrame(records)
        .groupby("feature", as_index=False)["lime_abs_weight"]
        .mean()
        .rename(columns={"lime_abs_weight": "lime_importance"})
        .sort_values("lime_importance", ascending=False)
        .reset_index(drop=True)
    )
    ranking["lime_rank"] = ranking.index + 1
    return ranking, perf_counter() - start_time

## 7. Agreement between SHAP and LIME

In [ ]:
def bootstrap_spearman_interval(first_ranks, second_ranks):
    generator = np.random.default_rng(RANDOM_SEED)
    sample_size = len(first_ranks)
    estimates = []
    for _ in range(BOOTSTRAP_REPLICATES):
        indices = generator.integers(0, sample_size, sample_size)
        if len(np.unique(first_ranks[indices])) < 3:
            continue
        estimates.append(spearmanr(first_ranks[indices], second_ranks[indices]).statistic)
    return float(np.percentile(estimates, 2.5)), float(np.percentile(estimates, 97.5))


def measure_shap_lime_agreement(model_name, shap_ranking, lime_ranking):
    """Inner join only. Features one method never scored are excluded, not given a filler rank."""
    aligned = shap_ranking.merge(lime_ranking, on="feature", how="inner")
    correlation = spearmanr(aligned["shap_rank"], aligned["lime_rank"])
    lower, upper = bootstrap_spearman_interval(
        aligned["shap_rank"].to_numpy(), aligned["lime_rank"].to_numpy()
    )

    shap_top = set(shap_ranking.head(STABILITY_TOP_K)["feature"])
    lime_top = set(lime_ranking.head(STABILITY_TOP_K)["feature"])

    return {
        "model": model_name,
        "features_compared": len(aligned),
        "spearman": correlation.statistic,
        "p_value": correlation.pvalue,
        "ci_lower": lower,
        "ci_upper": upper,
        "top10_overlap": len(shap_top & lime_top),
        "top10_jaccard": len(shap_top & lime_top) / len(shap_top | lime_top),
    }

## 8. LIME stability

In [ ]:
def measure_lime_stability(model_name, model, features_train, features_test):
    """Jaccard is computed per instance across seed pairs, then summarised over instances."""
    feature_names = features_train.columns.tolist()
    predict_probabilities = make_prediction_function(model, feature_names)
    stability_instances = features_test.sample(
        min(STABILITY_INSTANCE_COUNT, len(features_test)), random_state=RANDOM_SEED
    )
    explainers = [
        build_lime_explainer(features_train, RANDOM_SEED + run)
        for run in range(STABILITY_RUN_COUNT)
    ]

    per_instance = []
    for instance_index, (_, instance) in enumerate(stability_instances.iterrows()):
        top_feature_sets = []
        for explainer in explainers:
            explanation = explainer.explain_instance(
                instance.to_numpy(),
                predict_probabilities,
                num_features=STABILITY_TOP_K,
                num_samples=LIME_PERTURBATION_SAMPLES,
            )
            top_feature_sets.append(
                {feature_names[i] for i, _ in explanation.as_map()[1]}
            )

        pairwise = [
            len(a & b) / len(a | b) if (a | b) else 0.0
            for a, b in combinations(top_feature_sets, 2)
        ]
        per_instance.append({
            "model": model_name,
            "instance": instance_index,
            "jaccard": float(np.mean(pairwise)),
        })

    return pd.DataFrame(per_instance)


def summarise_stability(model_name, per_instance_frame):
    values = per_instance_frame["jaccard"].to_numpy()
    standard_error = values.std(ddof=1) / np.sqrt(len(values))
    return {
        "model": model_name,
        "stability_mean": values.mean(),
        "stability_sd": values.std(ddof=1),
        "ci_lower": values.mean() - 1.96 * standard_error,
        "ci_upper": values.mean() + 1.96 * standard_error,
        "instances": len(values),
        "runs": STABILITY_RUN_COUNT,
    }

## 9. SHAP against native tree importance

In [ ]:
def compare_shap_with_gini_importance(model_name, model, shap_ranking, feature_names):
    """Supports the TreeSHAP versus impurity importance claim. Tree models only."""
    if not hasattr(model, "feature_importances_"):
        return None

    native = (
        pd.DataFrame({
            "feature": feature_names,
            "gini_importance": model.feature_importances_,
        })
        .sort_values("gini_importance", ascending=False)
        .reset_index(drop=True)
    )
    native["gini_rank"] = native.index + 1

    aligned = shap_ranking.merge(native, on="feature", how="inner")
    correlation = spearmanr(aligned["shap_rank"], aligned["gini_rank"])

    shap_top = list(shap_ranking.head(TOP_FEATURES_REPORTED)["feature"])
    gini_top = list(native.head(TOP_FEATURES_REPORTED)["feature"])

    return {
        "model": model_name,
        "top_shap_features": ", ".join(shap_top),
        "top_gini_features": ", ".join(gini_top),
        "top5_overlap": len(set(shap_top) & set(gini_top)),
        "spearman": correlation.statistic,
        "p_value": correlation.pvalue,
    }

## 10. Experiment pipeline

In [ ]:
def run_experiment(dataset_name, encoded_frame, target_column):
    features_train, features_test, target_train, target_test, continuous_columns = (
        split_and_scale(encoded_frame, target_column)
    )

    # One shared sample, used by both explainers
    explanation_instances = features_test.sample(
        min(EXPLANATION_INSTANCE_COUNT, len(features_test)), random_state=RANDOM_SEED
    )
    effective_kernel_width = (
        LIME_KERNEL_WIDTH if LIME_KERNEL_WIDTH
        else float(np.sqrt(features_train.shape[1]) * 0.75)
    )

    print(f"\n=== {dataset_name} ===")
    print(f"Rows: {len(encoded_frame):,}   Encoded features: {features_train.shape[1]:,}")
    print(f"Continuous features standardised: {len(continuous_columns)}")
    print(f"Positive class share: {encoded_frame[target_column].mean():.3f}")
    print(f"Explanation instances: {len(explanation_instances)}")
    print(f"LIME perturbation samples: {LIME_PERTURBATION_SAMPLES}")
    print(f"LIME kernel width: {effective_kernel_width:.3f}")

    performance_rows = []
    agreement_rows = []
    stability_rows = []
    gini_rows = []
    stability_details = []
    shap_rankings = {}
    lime_rankings = {}

    for model_name, model in build_models(target_train).items():
        print(f"\nRunning {model_name} ...")

        train_start = perf_counter()
        model.fit(features_train, target_train)
        train_seconds = perf_counter() - train_start

        predict_start = perf_counter()
        predictions = model.predict(features_test)
        predict_seconds = perf_counter() - predict_start

        shap_ranking, shap_seconds = compute_shap_feature_ranking(
            model_name, model, features_train, explanation_instances
        )
        lime_ranking, lime_seconds = compute_lime_feature_ranking(
            model, features_train, explanation_instances
        )

        row = {"dataset": dataset_name, "model": model_name}
        row.update(score_predictions(target_test, predictions))
        row.update({
            "train_seconds": train_seconds,
            "predict_seconds": predict_seconds,
            "shap_seconds_per_instance": shap_seconds / len(explanation_instances),
            "lime_seconds_per_instance": lime_seconds / len(explanation_instances),
        })
        performance_rows.append(row)

        agreement = measure_shap_lime_agreement(model_name, shap_ranking, lime_ranking)
        agreement["dataset"] = dataset_name
        agreement_rows.append(agreement)

        per_instance = measure_lime_stability(model_name, model, features_train, features_test)
        per_instance["dataset"] = dataset_name
        stability_details.append(per_instance)

        stability = summarise_stability(model_name, per_instance)
        stability["dataset"] = dataset_name
        stability_rows.append(stability)

        gini = compare_shap_with_gini_importance(
            model_name, model, shap_ranking, features_train.columns
        )
        if gini is not None:
            gini["dataset"] = dataset_name
            gini_rows.append(gini)

        shap_rankings[model_name] = shap_ranking
        lime_rankings[model_name] = lime_ranking

        print(
            f"  accuracy={row['accuracy']:.3f}  recall={row['recall']:.3f}  "
            f"f1={row['f1_score']:.3f}  spearman={agreement['spearman']:.3f}  "
            f"stability={stability['stability_mean']:.3f}"
        )

    return {
        "performance": pd.DataFrame(performance_rows),
        "agreement": pd.DataFrame(agreement_rows),
        "stability": pd.DataFrame(stability_rows),
        "stability_detail": pd.concat(stability_details, ignore_index=True),
        "gini": pd.DataFrame(gini_rows),
        "shap_rankings": shap_rankings,
        "lime_rankings": lime_rankings,
    }

## 11. Run both datasets

This is the slow cell. Most of the time is spent in the LIME stability loop.

In [ ]:
telco_frame, telco_target = load_telco_dataset()
telco_results = run_experiment("Telco", telco_frame, telco_target)

adult_frame, adult_target = load_adult_dataset()
adult_results = run_experiment("Adult", adult_frame, adult_target)

all_results = {"Telco": telco_results, "Adult": adult_results}

## 12. Tables

Written to `outputs/`. Numbering follows the paper.

In [ ]:
def top_features_table(results_by_dataset, ranking_key, importance_column):
    rows = []
    for dataset_name, results in results_by_dataset.items():
        for model_name in MODEL_NAMES:
            ranking = results[ranking_key][model_name]
            for position in range(TOP_FEATURES_REPORTED):
                rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "rank": position + 1,
                    "feature": ranking.loc[position, "feature"],
                    "importance": ranking.loc[position, importance_column],
                })
    return pd.DataFrame(rows)


table_2_model_performance = pd.concat(
    [r["performance"] for r in all_results.values()], ignore_index=True
)[["dataset", "model", "accuracy", "precision", "recall", "f1_score"]].round(3)

table_3_shap_top_features = top_features_table(all_results, "shap_rankings", "shap_importance")
table_4_lime_top_features = top_features_table(all_results, "lime_rankings", "lime_importance")

table_5_shap_lime_agreement = pd.concat(
    [r["agreement"] for r in all_results.values()], ignore_index=True
)[["dataset", "model", "features_compared", "spearman", "ci_lower", "ci_upper",
   "p_value", "top10_overlap", "top10_jaccard"]].round(4)

table_6_lime_stability = pd.concat(
    [r["stability"] for r in all_results.values()], ignore_index=True
)[["dataset", "model", "stability_mean", "stability_sd", "ci_lower", "ci_upper",
   "instances", "runs"]].round(4)

table_7_runtime = pd.concat(
    [r["performance"] for r in all_results.values()], ignore_index=True
)[["dataset", "model", "train_seconds", "predict_seconds",
   "shap_seconds_per_instance", "lime_seconds_per_instance"]].round(4)

table_8_shap_vs_gini = pd.concat(
    [r["gini"] for r in all_results.values()], ignore_index=True
)[["dataset", "model", "top_shap_features", "top_gini_features",
   "top5_overlap", "spearman", "p_value"]].round(4)

# Significance of the stability difference between the two datasets
significance_rows = []
for model_name in MODEL_NAMES:
    telco_values = telco_results["stability_detail"].query("model == @model_name")["jaccard"]
    adult_values = adult_results["stability_detail"].query("model == @model_name")["jaccard"]
    test = mannwhitneyu(telco_values, adult_values, alternative="two-sided")
    significance_rows.append({
        "model": model_name,
        "telco_stability_mean": telco_values.mean(),
        "adult_stability_mean": adult_values.mean(),
        "difference": telco_values.mean() - adult_values.mean(),
        "mann_whitney_u": test.statistic,
        "p_value": test.pvalue,
    })
table_9_stability_significance = pd.DataFrame(significance_rows).round(4)

tables = {
    "table_2_model_performance": table_2_model_performance,
    "table_3_shap_top_features": table_3_shap_top_features,
    "table_4_lime_top_features": table_4_lime_top_features,
    "table_5_shap_lime_agreement": table_5_shap_lime_agreement,
    "table_6_lime_stability": table_6_lime_stability,
    "table_7_runtime": table_7_runtime,
    "table_8_shap_vs_gini": table_8_shap_vs_gini,
    "table_9_stability_significance": table_9_stability_significance,
}

for name, frame in tables.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    print(f"\n--- {name} ---")
    print(frame.to_string(index=False))

## 13. Figures

Three consolidated figures rather than one plot per model, so the paper fits the page limit.

In [ ]:
def plot_importance_panel(results_by_dataset, ranking_key, importance_column, title, filename):
    """One panel per dataset and model. Top 10 features by importance."""
    dataset_names = list(results_by_dataset.keys())
    figure, axes = plt.subplots(
        len(dataset_names), len(MODEL_NAMES),
        figsize=(15, 4.2 * len(dataset_names)),
    )
    axes = np.atleast_2d(axes)

    for row_index, dataset_name in enumerate(dataset_names):
        for column_index, model_name in enumerate(MODEL_NAMES):
            axis = axes[row_index, column_index]
            ranking = results_by_dataset[dataset_name][ranking_key][model_name].head(10)
            positions = np.arange(len(ranking))[::-1]

            axis.barh(positions, ranking[importance_column], color="#4c72b0", height=0.7)
            axis.set_yticks(positions)
            axis.set_yticklabels(ranking["feature"], fontsize=8)
            axis.set_title(f"{dataset_name} - {model_name}", fontsize=10)
            axis.tick_params(axis="x", labelsize=8)
            axis.spines["top"].set_visible(False)
            axis.spines["right"].set_visible(False)

    figure.suptitle(title, fontsize=13)
    figure.tight_layout(rect=[0, 0, 1, 0.97])
    figure.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_importance_panel(
    all_results, "shap_rankings", "shap_importance",
    "Mean absolute SHAP value, top 10 features",
    "figure_1_shap_feature_importance.png",
)

plot_importance_panel(
    all_results, "lime_rankings", "lime_importance",
    "Mean absolute LIME weight, top 10 features",
    "figure_2_lime_feature_importance.png",
)

In [ ]:
# Figure 3: distribution of per instance stability, all six model and dataset combinations
stability_all = pd.concat(
    [r["stability_detail"] for r in all_results.values()], ignore_index=True
)

groups = []
labels = []
for dataset_name in all_results:
    for model_name in MODEL_NAMES:
        subset = stability_all.query("dataset == @dataset_name and model == @model_name")
        groups.append(subset["jaccard"].to_numpy())
        labels.append(f"{dataset_name}\n{model_name}")

figure, axis = plt.subplots(figsize=(10, 5.5))
axis.boxplot(groups, labels=labels, widths=0.6)
axis.set_ylabel(f"Jaccard of top-{STABILITY_TOP_K} features across {STABILITY_RUN_COUNT} runs")
axis.set_ylim(0, 1)
axis.axhline(0.5, linestyle="--", linewidth=0.8, color="grey")
axis.set_title("LIME explanation stability per instance")
axis.tick_params(axis="x", labelsize=8)
axis.spines["top"].set_visible(False)
axis.spines["right"].set_visible(False)

figure.tight_layout()
figure.savefig(OUTPUT_DIR / "figure_3_lime_stability.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. Files written

In [ ]:
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)